In [33]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import joblib

from src.features.build_pipe_fe import build_pipe_fe
from src.features.new_features import TimeFeatureStrategy, AgeCategoryFeatureStrategy, HourCategoryFeatureStrategy
from sklearn.model_selection import train_test_split, GridSearchCV


In [2]:
df_transactions = pd.read_csv('../data/raw/transactions.csv', encoding='utf-8')
df_customers = pd.read_csv('../data/raw/customers.csv', encoding='utf-8')
df = pd.merge(
    df_transactions,
    df_customers,
    left_on='sender_id',
    right_on='customer_id',
    how='left'
)
target = 'fraud'
X = df.drop(columns=[target])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [3]:
features = [
    TimeFeatureStrategy(),
    AgeCategoryFeatureStrategy(),
    HourCategoryFeatureStrategy()
]

In [4]:
cat_cols = ['hour_category', 'age_category', 'account_type', 'gender', 'device_type']
num_cols = ['amount', 'age']

columns_drop = [
    'transaction_id', 'timestamp', 'sender_id', 'receiver_id',
    'customer_id', 'cpf', 'pix_key', 'hour_date', 'minute_date'
]

In [5]:
pipe_fe = build_pipe_fe(num_cols=num_cols, cat_cols=cat_cols, features=features, columns_drop=columns_drop)
X_resampled, y_resampled = pipe_fe.fit_resample(X_train, y_train)

In [13]:
pipe_test_fe = build_pipe_fe(num_cols=num_cols, cat_cols=cat_cols, features=features, columns_drop=columns_drop)
X_test_fe, y_test_fe = pipe_test_fe.fit_resample(X_test, y_test)

---

## Select Model

In [28]:
models_params = [
    {
        'name': 'LogisticRegression',
        'model': LogisticRegression(max_iter=1000, class_weight='balanced'),
        'param_grid': {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l2'],
            'solver': ['lbfgs', 'liblinear']
        }
    },
    {
        'name': 'RandomForest',
        'model': RandomForestClassifier(class_weight='balanced'),
        'param_grid': {
            'n_estimators': [100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5]
        }
    }
]

In [31]:

best_score = -1
best_model = None
best_name = None
best_params = None

for mp in models_params:
    grid = GridSearchCV(
        estimator=mp['model'],
        param_grid=mp['param_grid'],
        cv=3,
        scoring='f1'
    )
    grid.fit(X_resampled, y_resampled)

    if grid.best_score_ > best_score:
        best_score = grid.best_score_
        best_model = grid.best_estimator_
        best_name = mp['name']
        best_params = grid.best_params_

In [32]:
print(f"Melhor modelo: {best_name}")
print(f"Melhores parâmetros: {best_params}")
print(f"Melhor F1: {best_score:.4f}")

Melhor modelo: RandomForest
Melhores parâmetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Melhor F1: 0.8787


In [35]:

joblib.dump(best_model, f'../data/models/best_model.pkl')
joblib.dump(best_params, f'../data/models/best_params.pkl')

['../data/models/best_params.pkl']